# 4. 动态评估法
贝尔曼方程只对小问题有效。在实践中，当状态和行动模式的数量稍有增加，这种方法就会变得难以使用。对于这样的情况应运而生了动态规划法

## 4.1 动态规划法和策略评估
强化学习的问题通常涉及：策略评估和策略控制  
策略评估是求给定策略 $\pi$ 的价值函数 $v_\pi(s)$ 和 $q_\pi(s,a)$，策略控制是控制策略并将其调整为最优策略
强化学习的最终目标是策略控制，但为实现这一目标需要进行策略评估，现在通过动态规划法（DP）的算法进行策略评估

### 4.1.1 动态规划法简介

使用DP的方法也是从贝尔曼方程通过“更新式”衍生而来，通过用估计值来改进另一个估计值的自举法来完成更迭，这种算法被称为迭代策略评估
$$
V_{k+1}(s)=\sum_{a,s'} \pi(a|s)p(s'|s,a) \left\{r(s,a,s') + \gamma V_k(s')\right\}
$$

## 4.2 更大的问题

### 4.2.1 GridWorld类的实现

In [ ]:
import numpy as np

class GridWorld:
    def __init__(self):
        self.action_space = [0, 1, 2, 3]
        self.action_meaning = {
            0: 'UP',
            1: 'DOWN',
            2: 'LEFT',
            3: 'RIGHT',
        }

        self.reward_map = np.array(
            [[0, 0, 0, 1.0],
             [0, None, 0, -1.0],
             [0, 0, 0, 0]]
        )
        self.goal_state = (0, 3)
        self.wall_state = (1, 1)
        self.start_state = (2, 0)
        self.agent_state = self.start_state

    @property
    def height(self):
        return len(self.reward_map)
    
    @property
    def width(self):
        return len(self.reward_map[0])

    @property
    def shape(self):
        return self.reward_map.shape

    def actions(self):
        return self.action_space # [0, 1, 2, 3]

    def states(self):
        for h in range(self.height):
            for w in range(self.width):
                yield(h, w)

In [10]:
env = GridWorld()

print(env.height)
print(env.width)
print(env.shape)

for action in env.actions():
    print(action)

print('===')

for state in env.states():
    print(state)

3
4
(3, 4)
0
1
2
3
===
(0, 0)
(0, 1)
(0, 2)
(0, 3)
(1, 0)
(1, 1)
(1, 2)
(1, 3)
(2, 0)
(2, 1)
(2, 2)
(2, 3)


In [ ]:
import numpy as np

class GridWorld:
    def __init__(self):
        self.action_space = [0, 1, 2, 3]
        self.action_meaning = {
            0: 'UP',
            1: 'DOWN',
            2: 'LEFT',
            3: 'RIGHT',
        }

        self.reward_map = np.array(
            [[0, 0, 0, 1.0],
             [0, None, 0, -1.0],
             [0, 0, 0, 0]]
        )
        self.goal_state = (0, 3)
        self.wall_state = (1, 1)
        self.start_state = (2, 0)
        self.agent_state = self.start_state

    @property
    def height(self):
        return len(self.reward_map)
    
    @property
    def width(self):
        return len(self.reward_map[0])

    @property
    def shape(self):
        return self.reward_map.shape

    def actions(self):
        return self.action_space # [0, 1, 2, 3]

    def states(self):
        for h in range(self.height):
            for w in range(self.width):
                yield(h, w)

    def next_state(self, state, action):
        # 移动目的地的计算
        action_move_map = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        move = action_move_map[action]
        next_state = (state[0] + move[0], state[1] + move[1])
        ny, nx = next_state

        if nx < 0 or nx >= self.width or ny < 0 or ny >= self.height: # 判断是否出界
            next_state = state
        elif next_state == self.wall_state: # 判断是否撞墙
            next_state = state
        return next_state # 返回下一状态
        # 因为这个人物中状态迁移是确定性的，所以直接返回

    def reward(self, state, action, next_state):
        return self.reward_map[next_state]

env = GridWorld()
V = {}

for state in env.states(): # 遍历网格所有位置
    V[state] = 0 # 每个位置价值初始化为0

state = (1, 2)
print(V[state]) # 输出状态(1,2)的价值函数

0


In [ ]:
from collections import defaultdict

env = GridWorld()
V = defaultdict(lambda:0) # 任何未定义的状态，自动返回0

state = (1, 2)
print(V[state])

0


实现随机性策略，按照均匀随机往四周的概率都是1/4

In [ ]:
pi = defaultdict(lambda: {0:0.25, 1:0.25, 2:0.25, 3:0.25})

state = (0, 1)
print(pi[state])

{0: 0.25, 1: 0.25, 2: 0.25, 3: 0.25}


In [ ]:
def eval_onestep(pi, V, env, gamma=0.9):
    '''
    pi:策略
    V:价值函数
    env:环境
    gamma:折现率
    '''
    for state in env.states(): # 访问各个状态
        if state == env.goal_state: # 目的地的价值函数总是为0
            V[state] = 0
            continue
        
        action_probs = pi[state]
        new_V = 0

        for action, action_prob in action_probs.items():
            next_state = env.next_state(state, action)
            r = env.reward(state, action, next_state)

            new_V += action_prob * (r + gamma * V[next_state])
        V[state] = new_V
return V